# System Prompt Switching

System prompt switching is the foundational technique for implementing agent modes. The idea is simple: the mode an agent operates in is encoded in a system prompt, and switching modes means replacing that system prompt with a different one.

This simplicity is precisely why it is the most widely deployed mode-implementation technique in production AI systems today. It requires no specialized frameworks, works with every LLM provider, and maps directly onto how models process instructions at inference time. Every major chatbot and copilot product - whether built on Claude, GPT-4, or Gemini - uses this technique at its core.

### When to use system prompt switching
System prompt switching is the right choice when:
- The mode is known before the conversation begins
- Simplicity is a priority — no dynamic classification is needed
- Mode changes happen at session boundaries or on explicit user request
- The application layer, not the LLM itself, decides which mode is active

For use cases where the LLM needs to detect and switch modes dynamically based on conversation content, see the dynamic mode injection notebook.

In [1]:
import os
from typing import Annotated, Sequence, Optional
from dataclasses import dataclass, field

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict

### Initialize the language model
We use GPT-4o-mini throughout this notebook. It follows system prompt instructions accurately, which makes it a reliable tool for observing how different system prompts shape behavior from the same underlying model.

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0)
print("LLM initialized:", llm.model_name)

LLM initialized: gpt-4o-mini


## Section 1: Simple system prompt switching
The most direct form of mode management is a dictionary that maps mode names to system prompt strings. When a user message arrives, the agent looks up the right system prompt and injects it as the first message in the LLM call.

This is exactly how most production chatbots work under the hood. A customer support platform might load a different system prompt depending on which product queue the user contacted. A coding assistant might switch from a "review" prompt to a "documentation" prompt based on a button the user clicked. The mechanism is always the same: swap the system prompt.

The three modes below represent a common behavioral spectrum. Each defines a completely different behavioral contract for the same underlying model.

In [3]:
# Each key is a mode name; each value is the full system prompt for that mode.
# This dictionary is the only structure needed for basic system prompt switching.
MODE_SYSTEM_PROMPTS: dict[str, str] = {
    "chat": (
        "You are a friendly, conversational AI assistant. "
        "Keep responses warm, approachable, and concise. "
        "Prefer plain language over technical jargon. "
        "Match the user's energy and tone."
    ),
    "research": (
        "You are a rigorous research assistant. "
        "Provide detailed, well-structured analyses with clear reasoning. "
        "Always surface trade-offs, limitations, and alternative perspectives. "
        "Cite your reasoning explicitly. Use headings and bullet points for clarity."
    ),
    "task_execution": (
        "You are a precise task-execution agent. "
        "Respond only with the concrete steps or output required. "
        "Skip pleasantries and background explanation unless asked. "
        "Use numbered lists for sequential steps. "
        "State assumptions explicitly before executing."
    ),
}

print("Defined modes:", list(MODE_SYSTEM_PROMPTS.keys()))

Defined modes: ['chat', 'research', 'task_execution']


The agent class below wraps the LLM with two responsibilities: tracking conversation history across turns, and injecting the correct system prompt on every call. The system prompt is intentionally stored outside the conversation history — this separation is the key design decision that makes mode switching possible. History accumulates turn by turn, while the system prompt can be swapped at any point without touching that history.

Clearing history on mode switch (`clear_history=True`) is the default and safest behavior, since it prevents the previous mode's conversational context from contaminating the new mode. In some use cases - like escalating from chat to research mid-session — we may want to preserve context, which is why `clear_history` is a parameter rather than hardcoded.

In [4]:
class SystemPromptSwitchingAgent:
    """An agent that changes behavior by swapping its system prompt."""

    def __init__(self, initial_mode: str = "chat"):
        self.current_mode = initial_mode
        # History is stored without any system prompt — the system prompt is injected fresh each turn
        self.conversation_history: list[BaseMessage] = []

    def set_mode(self, mode: str, clear_history: bool = True) -> None:
        """Switch to a different mode.

        Args:
            mode: Target mode name. Must be a key in MODE_SYSTEM_PROMPTS.
            clear_history: When True (default), resets conversation history on switch.
        """
        if mode not in MODE_SYSTEM_PROMPTS:
            raise ValueError(f"Unknown mode '{mode}'. Available: {list(MODE_SYSTEM_PROMPTS)}")

        previous = self.current_mode
        self.current_mode = mode

        if clear_history:
            # Reset so the new mode starts without any prior conversation context
            self.conversation_history = []

        print(f"[Mode switched] {previous} \u2192 {mode}")

    def respond(self, user_message: str) -> str:
        """Generate a response in the currently active mode."""
        # Fetch the system prompt for the active mode
        system_prompt = MODE_SYSTEM_PROMPTS[self.current_mode]

        # Build the message list: system prompt first, then prior history, then the new message
        messages = [SystemMessage(content=system_prompt)]
        messages.extend(self.conversation_history)
        messages.append(HumanMessage(content=user_message))

        response = llm.invoke(messages)

        # Persist this turn to history so context is available in subsequent turns
        self.conversation_history.append(HumanMessage(content=user_message))
        self.conversation_history.append(AIMessage(content=response.content))

        return response.content

The simplest way to understand the effect of system prompt switching is to send the same question through every mode. The model, the API call structure, and the input text are identical - only the system prompt differs.

In [5]:
agent = SystemPromptSwitchingAgent()
question = "What is retrieval-augmented generation and when should I use it?"

for mode in ["chat", "research", "task_execution"]:
    print("=" * 60)
    print(f"MODE: {mode.upper()}")
    print("=" * 60)
    agent.set_mode(mode)
    response = agent.respond(question)
    # Truncate long responses so the behavioral difference stands out
    print(response[:500] + ("..." if len(response) > 500 else ""))
    print()

MODE: CHAT
[Mode switched] chat → chat
Retrieval-augmented generation (RAG) is a method that combines two powerful techniques: retrieving relevant information from a database or knowledge base and then using that information to generate responses. It’s like having a smart assistant that not only knows a lot but can also look up specific details to give you the best answer.

You might want to use RAG when you need accurate, up-to-date information or when your questions are complex and require specific details. It’s great for tasks lik...

MODE: RESEARCH
[Mode switched] chat → research
# Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation (RAG) is a hybrid approach that combines the strengths of information retrieval and generative models to enhance the quality and relevance of generated text. This method is particularly useful in scenarios where the model needs to produce contextually accurate and factually grounded responses.

## Key Components of RAG

1. **Retrieval C

The responses reflect three distinct behavioral contracts even though the underlying model and question are identical:
- **Chat mode** gives a warm, accessible explanation without technical overhead
- **Research mode** structures the answer into sections and surfaces trade-offs and limitations
- **Task execution mode** skips background and delivers actionable criteria in a numbered list

The model's knowledge is the same in all three cases. What changes is how it scopes and presents that knowledge - determined entirely by the system prompt.

## Section 2: Structured mode definitions
Storing system prompts as plain strings in a dictionary is perfectly valid for a small, stable set of modes. As the number of modes grows, raw strings become difficult to maintain: tweaking tone or constraints requires hunting through multi-line string literals, there is no way to inspect individual behavioral parameters programmatically, and there is no natural place to attach metadata like a version number for audit logs.

A lightweight improvement is to store mode definitions as structured data and let each definition render its own system prompt. This way, each behavioral parameter - tone, format, constraints - becomes a named field we can read, modify, and log independently, without touching prompt templates.

In [6]:
@dataclass
class ModeDefinition:
    """Structured specification for a single agent mode.

    Each field maps to a specific behavioral directive in the rendered system prompt.
    The version field is used in session logs to tie a conversation to an exact prompt snapshot.
    """
    name: str
    role_description: str                                  # What the agent is and does
    tone: str                                              # Communication style
    response_format: str                                   # Expected output structure
    capabilities: list[str] = field(default_factory=list) # Behaviors explicitly encouraged
    constraints: list[str] = field(default_factory=list)  # Behaviors explicitly forbidden
    version: str = "1.0"                                   # Tracks prompt changes over time

    def to_prompt(self) -> str:
        """Render this definition into a system prompt string ready for the LLM."""
        lines = [
            f"Role: {self.role_description}",
            f"Tone: {self.tone}",
            f"Response format: {self.response_format}",
        ]

        # Only add a capabilities section if any were explicitly defined
        if self.capabilities:
            lines.append("You may: " + "; ".join(self.capabilities) + ".")

        # Only add a constraints section if any were explicitly defined
        if self.constraints:
            lines.append("You must not: " + "; ".join(self.constraints) + ".")

        return "\n".join(lines)

**`ModeDefinition`** is a plain dataclass - no extra dependencies and no base classes required.

The key design choice is placing `to_prompt()` directly on the dataclass rather than in a separate builder class. The definition owns its own rendering logic, keeping usage simple: `mode.to_prompt()` always returns the current prompt. When a mode changes, we update the field values and call `to_prompt()` again - no need to touch a separate builder or template.

Separating `capabilities` (allowed behaviors) from `constraints` (prohibited behaviors) follows a pattern from production safety frameworks. Explicit prohibition is more reliable than implicit omission: the model needs to be told "never promise refunds outside policy," not merely told what it *is* allowed to do.

In [7]:
# Three distinct mode definitions — same dataclass structure, entirely different prompts
research_mode = ModeDefinition(
    name="research",
    role_description="A rigorous research analyst who synthesizes information from multiple sources.",
    tone="precise, evidence-driven, and neutral",
    response_format="structured sections: Summary, Key Findings, Trade-offs, Limitations",
    capabilities=[
        "cite reasoning explicitly",
        "highlight confidence levels",
        "surface contradictions between sources",
    ],
    constraints=[
        "do not assert facts without evidence",
        "do not recommend specific vendors",
    ],
    version="1.0",
)

support_mode = ModeDefinition(
    name="customer_support",
    role_description="A support specialist who resolves product issues and answers account questions.",
    tone="warm, empathetic, and solution-focused",
    response_format="short paragraphs; numbered steps for troubleshooting",
    capabilities=["ask clarifying questions", "escalate to a human agent when needed"],
    constraints=[
        "never promise refunds outside policy",
        "never share account details unless verified",
    ],
    version="1.0",
)

code_reviewer_mode = ModeDefinition(
    name="code_reviewer",
    role_description="A senior software engineer reviewing code for quality, security, and correctness.",
    tone="direct, technical, and constructive \u2014 critique the code, not the author",
    response_format="Issue / Severity / Fix format, followed by a corrected code block",
    capabilities=["identify security vulnerabilities", "suggest readability improvements"],
    constraints=["never make personal comments", "never approve code with critical security issues"],
    version="1.0",
)

# Render and inspect each system prompt to verify the output
for mode_def in [research_mode, support_mode, code_reviewer_mode]:
    print(f"--- {mode_def.name.upper()} (v{mode_def.version}) ---")
    print(mode_def.to_prompt())
    print()

--- RESEARCH (v1.0) ---
Role: A rigorous research analyst who synthesizes information from multiple sources.
Tone: precise, evidence-driven, and neutral
Response format: structured sections: Summary, Key Findings, Trade-offs, Limitations
You may: cite reasoning explicitly; highlight confidence levels; surface contradictions between sources.
You must not: do not assert facts without evidence; do not recommend specific vendors.

--- CUSTOMER_SUPPORT (v1.0) ---
Role: A support specialist who resolves product issues and answers account questions.
Tone: warm, empathetic, and solution-focused
Response format: short paragraphs; numbered steps for troubleshooting
You may: ask clarifying questions; escalate to a human agent when needed.
You must not: never promise refunds outside policy; never share account details unless verified.

--- CODE_REVIEWER (v1.0) ---
Role: A senior software engineer reviewing code for quality, security, and correctness.
Tone: direct, technical, and constructive — cri

The rendered prompts are clean, inspectable, and easy to audit. When a mode produces unexpected behavior, we can check `research_mode.tone` or `support_mode.constraints` directly in code - no string parsing required. The `version` field provides a human-readable label that can be logged alongside every session, making it straightforward to reproduce the exact behavioral specification active at any point in time.

Now let's confirm that structured definitions produce the same behavioral differentiation we saw with raw strings.

In [8]:
question = "Why is my application running slowly?"

for mode_def in [support_mode, research_mode]:
    # Render the system prompt from the structured definition
    system_prompt = mode_def.to_prompt()

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question),
    ])

    print(f"=== {mode_def.name.upper()} MODE (v{mode_def.version}) ===")
    print(response.content[:400] + ("..." if len(response.content) > 400 else ""))
    print()

=== CUSTOMER_SUPPORT MODE (v1.0) ===
I understand how frustrating it can be when an application runs slowly. Let's work together to troubleshoot this issue. Here are a few steps you can take to identify the problem:

1. **Check Your Internet Connection**: Ensure that you have a stable internet connection. You can try restarting your router or switching to a different network if possible.

2. **Close Unnecessary Applications**: If you...

=== RESEARCH MODE (v1.0) ===
### Summary
The performance of an application can be influenced by a variety of factors, including hardware limitations, software inefficiencies, network issues, and user behavior. Identifying the root cause of slow performance requires a systematic analysis of these potential factors.

### Key Findings
1. **Hardware Limitations**:
   - **CPU and Memory**: Insufficient CPU power or RAM can lead to...



The support mode opens with empathy and guides the user through concrete troubleshooting steps. The research mode frames the same question analytically and organizes the answer around categories of cause. Same model, same question, same knowledge - different behavioral lens.

## Section 3: Stateful mode switching with LangGraph
The examples so far use system prompt switching between sessions - the mode is set once before the conversation starts. Many real deployments also need to support mode switching within a session: a user typing `/mode research` mid-conversation, or a supervisor escalating an ongoing chat to a higher-autonomy mode.

LangGraph's explicit state management makes this straightforward. The graph holds both the conversation history and the currently active mode as state fields, so the mode persists across turns without re-evaluation. A dedicated routing node inspects each incoming message for a mode-switch command before passing it to the LLM - mode transitions and regular responses travel through distinct, inspectable graph paths.

In [9]:
class AgentState(TypedDict):
    """State that persists across all turns of the conversation.

    Fields:
        messages: Full conversation history, accumulated turn by turn using LangGraph's add_messages reducer (new messages are appended, not replaced).
        current_mode: The active mode name — initialized once and updated by /mode commands.
    """
    messages: Annotated[Sequence[BaseMessage], add_messages]
    current_mode: str

With `AgentState` defined, we need a flat lookup table mapping mode names to rendered system prompt strings. The graph nodes query this table on every turn, so prompt rendering should happen once at startup - not on every incoming message. We build the registry from `ModeDefinition` instances, ensuring the structured definition and the rendered prompt are always in sync.

In [10]:
# Collect all mode definitions in one place — add new modes here as the application grows
MODE_DEFINITIONS: dict[str, ModeDefinition] = {
    "chat": ModeDefinition(
        name="chat",
        role_description="A helpful, conversational assistant for general questions.",
        tone="friendly and approachable",
        response_format="concise prose, under 100 words unless more detail is needed",
    ),
    "research": research_mode,
    "customer_support": support_mode,
    "code_review": code_reviewer_mode,
}

# Render all prompts once at startup; MODE_REGISTRY[mode_name] gives the prompt string
MODE_REGISTRY: dict[str, str] = {
    name: defn.to_prompt()
    for name, defn in MODE_DEFINITIONS.items()
}

# Sanity check: show registered modes, prompt lengths, and current versions
print("Registered modes:")
for name, prompt in MODE_REGISTRY.items():
    version = MODE_DEFINITIONS[name].version
    print(f"  {name:<22} \u2014 {len(prompt):3d} chars  (v{version})")

Registered modes:
  chat                   — 173 chars  (v1.0)
  research               — 405 chars  (v1.0)
  customer_support       — 371 chars  (v1.0)
  code_review            — 420 chars  (v1.0)


The graph has two nodes. The **routing node** runs first on every incoming message: it checks whether the message is a `/mode <name>` command, and if it is, handles the switch itself and emits a confirmation message without ever calling the LLM. If the message is not a switch command, the router returns the state unchanged and the graph proceeds to the **response node**.

Keeping mode-switching logic out of the LLM call has three benefits: it makes transitions deterministic (no risk of the model misinterpreting a switch command), it keeps latency low (no LLM call for a mode switch), and it keeps the routing behavior fully inspectable as plain Python.

In [11]:
def route_or_respond(state: AgentState) -> dict:
    """Detect /mode commands and switch modes, or pass through for LLM response.

    Mode switch commands follow the pattern: /mode <name>
    All other messages pass through unchanged.
    """
    last_message = state["messages"][-1]

    # Only inspect human messages — AI messages pass through without modification
    if not isinstance(last_message, HumanMessage):
        return state

    text = last_message.content.strip()

    if text.startswith("/mode "):
        # Extract the mode name that follows the "/mode " prefix
        requested_mode = text[6:].strip()

        if requested_mode in MODE_REGISTRY:
            # Valid mode — emit a confirmation message and update the mode in state
            description = MODE_DEFINITIONS[requested_mode].role_description
            confirmation = AIMessage(
                content=f"[Switched to **{requested_mode}** mode] {description}"
            )
            return {"messages": [confirmation], "current_mode": requested_mode}

        else:
            # Unknown mode — inform the user without crashing or changing the mode
            available = ", ".join(MODE_REGISTRY.keys())
            error_msg = AIMessage(
                content=f"Unknown mode '{requested_mode}'. Available modes: {available}"
            )
            return {"messages": [error_msg], "current_mode": state["current_mode"]}

    # Not a mode command — return unchanged so the respond node handles the message
    return state

The response node generates the LLM reply. It reads `current_mode` from state to look up the right system prompt, prepends it to the full message history, and invokes the model. The system prompt is prepended fresh on every call — it is never stored inside the message history itself. This keeps the history clean (showing only what the user and agent actually said) and makes it easy to change the system prompt without touching stored conversation data.

In [12]:
def respond(state: AgentState) -> dict:
    """Generate an LLM response using the active mode's system prompt."""
    # Look up the rendered system prompt for the current mode
    system_prompt = MODE_REGISTRY.get(state["current_mode"], MODE_REGISTRY["chat"])

    # Inject the system prompt at position 0; the conversation history follows it
    messages = [SystemMessage(content=system_prompt)] + list(state["messages"])
    response = llm.invoke(messages)

    return {"messages": [response]}


def should_respond(state: AgentState) -> str:
    """Decide whether to call the respond node or end the turn.

    If route_or_respond already produced an AIMessage (a mode-switch confirmation), we skip the respond node to avoid generating a second response on the same turn.
    """
    last = state["messages"][-1]

    # Mode-switch confirmation is already an AIMessage — no LLM call needed
    if isinstance(last, AIMessage):
        return END

    return "respond"

The graph wires the two nodes together with a conditional edge out of the routing node. After `route_or_respond` runs, `should_respond` checks the last message in state: if it is already an `AIMessage` (the router handled a mode switch), the turn ends immediately; otherwise, the graph proceeds to the `respond` node. This produces a clean single-entry, dual-exit flow where mode switches and normal responses are handled as separate paths.

In [13]:
graph_builder = StateGraph(AgentState)

graph_builder.add_node("route_or_respond", route_or_respond)
graph_builder.add_node("respond", respond)

# Every turn enters at the routing node
graph_builder.add_edge(START, "route_or_respond")

# After routing, decide whether an LLM call is needed
graph_builder.add_conditional_edges("route_or_respond", should_respond)

# After the LLM responds, the turn is complete
graph_builder.add_edge("respond", END)

agent_graph = graph_builder.compile()
print("Agent graph compiled successfully")

Agent graph compiled successfully


The helper function below manages the state dictionary between turns, which is the minimal state-threading pattern for a LangGraph agent in a script environment. We pass the previous state into each `invoke()` call and store the returned state for the next turn. The active mode is visible in `state["current_mode"]` after every turn, making it easy to confirm which mode is active at any point in the log.

In [14]:
def send_message(user_message: str, state: dict) -> tuple[str, dict]:
    """Send one message, return the agent's reply and the updated state."""
    # Append the new human message to the existing history before invoking the graph
    state["messages"] = list(state.get("messages", [])) + [HumanMessage(content=user_message)]
    result = agent_graph.invoke(state)
    # The last message in result is always the agent's reply for this turn
    return result["messages"][-1].content, result


# Initialize the agent in chat mode with an empty conversation
state = {"messages": [], "current_mode": "chat"}

conversation = [
    ("Hello! What can you help me with?",              "Turn 1 \u2014 chat mode"),
    ("/mode research",                                  "Turn 2 \u2014 switch modes"),
    ("What are the trade-offs between SQL and NoSQL?",  "Turn 3 \u2014 research query"),
    ("/mode code_review",                               "Turn 4 \u2014 switch modes"),
    ("Review this: def divide(a, b): return a / b",    "Turn 5 \u2014 code review task"),
]

for user_input, label in conversation:
    print(f"[{label}]")
    print(f"User:  {user_input}")
    reply, state = send_message(user_input, state)
    print(f"Agent: {reply[:350]}{'...' if len(reply) > 350 else ''}")
    print(f"Active mode: {state['current_mode']}")
    print()

[Turn 1 — chat mode]
User:  Hello! What can you help me with?
Agent: Hello! I can assist you with a variety of topics, including answering questions, providing information, offering advice, and helping with problem-solving. Whether you need help with general knowledge, tips on a specific subject, or just want to chat, I’m here for you! What do you need assistance with today?
Active mode: chat

[Turn 2 — switch modes]
User:  /mode research
Agent: [Switched to **research** mode] A rigorous research analyst who synthesizes information from multiple sources.
Active mode: research

[Turn 3 — research query]
User:  What are the trade-offs between SQL and NoSQL?
Agent: ### Summary
The choice between SQL (Structured Query Language) and NoSQL (Not Only SQL) databases involves several trade-offs that can significantly impact application performance, scalability, and data integrity. SQL databases are typically relational and structured, while NoSQL databases offer more flexibility in terms of dat

The agent carries the active mode across turns without re-detecting it from the conversation. Mode transitions are instant because the router intercepts `/mode` commands before they reach the LLM — there is zero LLM cost for a mode switch.

## Section 4: Preventing prompt dilution in long conversations
In long conversations, the system prompt's influence can gradually weaken. This happens because the model assigns attention to all tokens in its context window — and as the conversation history grows, the system prompt tokens represent a smaller and smaller fraction of the total. The model may begin mirroring the tone of recent user messages, drift from the required response format, or soften constraints that were specified in the system prompt.

The most practical mitigation is to inject a brief **mode reminder** into the message history at regular intervals. The reminder does not repeat the full system prompt — it simply reinforces the behavioral contract with a single sentence placed close to the recent context where the model's attention is highest.

In [15]:
def inject_mode_reminder_if_needed(
    history: list[BaseMessage],
    mode_name: str,
    every_n_turns: int = 6,
) -> list[BaseMessage]:
    """Append a mode reminder to history at regular intervals.

    A 'turn' is one human message and one AI response (two messages in history).

    Args:
        history: Current conversation history.
        mode_name: Name of the active mode to reinforce.
        every_n_turns: How often to inject a reminder. Defaults to every 6 turns.

    Returns:
        A new history list with a reminder SystemMessage appended if the threshold is met.
        Returns the original list unchanged if the threshold is not yet reached.
    """
    # Count completed human turns (each HumanMessage represents one user turn)
    human_turns = sum(1 for m in history if isinstance(m, HumanMessage))

    # Fire at turns 6, 12, 18, ... — not on every turn
    if human_turns > 0 and human_turns % every_n_turns == 0:
        reminder = SystemMessage(
            content=(
                f"[Mode reminder: You are in {mode_name.upper()} mode. "
                f"Continue applying all mode constraints and the required response format. "
                f"Do not drift from the behavioral contract defined in your system prompt.]"
            )
        )
        # Return a new list — do not mutate the original
        return list(history) + [reminder]

    return history

**`inject_mode_reminder_if_needed`** is a pure function — it returns a new list and never mutates the input. The `human_turns % every_n_turns == 0` condition fires at turns 6, 12, 18, and so on. Six turns is a practical default: close enough to the start of potential drift, infrequent enough to avoid cluttering the context window.

The reminder is injected as a `SystemMessage`, not a `HumanMessage` or `AIMessage`. This matters: most LLM providers treat `SystemMessage` entries anywhere in the message list as authoritative instructions. Injecting it as a `HumanMessage` would make it appear to come from the user, which reduces its authority and could confuse the model about who issued the instruction.

Let's confirm the reminder has a measurable effect by comparing LLM responses with and without it, simulating an agent that has just completed six turns of casual, off-format conversation.

In [16]:
# Build a 6-turn history filled with casual, off-format exchanges to simulate drift
simulated_history: list[BaseMessage] = []
for i in range(1, 7):
    simulated_history.append(HumanMessage(content=f"Quick tip #{i} for RAG?"))
    # These AI replies are deliberately casual and unstructured — the opposite of research format
    simulated_history.append(AIMessage(content=f"Sure! Here's a quick tip #{i}: use chunking wisely."))

# The actual question we want to evaluate the format of
test_question = "Give me a structured analysis of hybrid search vs. dense-only retrieval."

research_system_prompt = research_mode.to_prompt()

# --- Without reminder: system prompt + 6 casual turns + test question ---
messages_without = (
    [SystemMessage(content=research_system_prompt)]
    + simulated_history
    + [HumanMessage(content=test_question)]
)
response_without = llm.invoke(messages_without)

print("WITHOUT mode reminder (after 6 casual turns):")
print("-" * 50)
print(response_without.content[:600])

print()

# --- With reminder: inject the reminder after turn 6, then append the test question ---
# Pass the history (without the test question) to the reminder function
history_with_reminder = inject_mode_reminder_if_needed(
    simulated_history,    # 6 turns of casual history
    mode_name="research",
    every_n_turns=6,      # Threshold is exactly met — reminder fires
)
messages_with = (
    [SystemMessage(content=research_system_prompt)]
    + history_with_reminder
    + [HumanMessage(content=test_question)]
)
response_with = llm.invoke(messages_with)

print("WITH mode reminder (injected at turn 6):")
print("-" * 50)
print(response_with.content[:600])

WITHOUT mode reminder (after 6 casual turns):
--------------------------------------------------
### Summary
Hybrid search and dense-only retrieval are two distinct approaches to information retrieval in the context of search engines and natural language processing. Hybrid search combines traditional keyword-based methods with modern vector-based techniques, while dense-only retrieval relies solely on vector representations of data for retrieving relevant information. This analysis explores the key findings, trade-offs, and limitations of both approaches.

### Key Findings

1. **Hybrid Search**:
   - **Mechanism**: Combines traditional keyword search (using inverted indices) with dense ve

WITH mode reminder (injected at turn 6):
--------------------------------------------------
### Summary
Hybrid search and dense-only retrieval are two approaches used in information retrieval systems, particularly in the context of natural language processing and search engines. Hybrid search combine

System prompt switching is the production standard for implementing agent modes because it is simple, transparent, and universally supported across every LLM provider and framework.

**Core design principles:**

1. **Encode modes as data, not code** — Store system prompts in dictionaries or dataclasses, not scattered across code files. This makes modes reviewable, version-trackable, and changeable without code deployments.
2. **Keep history separate from mode config** — Conversation history and the active system prompt serve different purposes. Storing them separately lets us switch modes, clear history, or preserve context independently of each other.
3. **Handle mode switching at the application layer** — For deterministic mode switching triggered by user commands or UI events, route it in application logic rather than asking the LLM to detect switch intent. This is faster, cheaper, and more reliable.
4. **Inject mode reminders in long conversations** — Periodic reinforcement prevents the model from drifting from the mode's behavioral contract as conversation history grows and dilutes the system prompt's relative weight.

**When to move beyond system prompt switching:**
- When mode selection must be inferred from conversation context → **dynamic mode injection**
- When modes need to be portable, formally validated artifacts → **mode as skill**
- When multiple modes need to operate simultaneously on different sub-tasks → **mode composition**